[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/28_moe.ipynb)

# 🔴 Hard: Mixture of Experts (MoE)

Implement a **Mixture of Experts** layer (Mixtral / Switch Transformer style).

### Signature
```python
class MixtureOfExperts(nn.Module):
    def __init__(self, d_model, d_ff, num_experts, top_k=2): ...
    def forward(self, x: Tensor) -> Tensor:
        # x: (B, S, D) -> (B, S, D)
```

### Architecture
- `self.router`: `nn.Linear(d_model, num_experts)` — gating network
- `self.experts`: `nn.ModuleList` of MLPs `(Linear→ReLU→Linear)`
- For each token: select top-k experts, compute weighted sum of their outputs

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.3 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn

In [7]:
from pygments.token import Token
# ✏️ YOUR IMPLEMENTATION HERE

class MixtureOfExperts(nn.Module):
    def __init__(self, d_model, d_ff, num_experts, top_k=2):
      super().__init__()
      # router + experts
      self.d_model = d_model
      self.d_ff = d_ff
      self.num_experts = num_experts
      self.top_k = top_k

      self.router = nn.Linear(d_model, num_experts)
      self.experts = nn.ModuleList(
          nn.Sequential(
              nn.Linear(d_model, d_ff),
              nn.ReLU(),
              nn.Linear(d_ff, d_model)
          ) for _ in range(num_experts)
      )

    def forward(self, x):
      B, S, d_model = x.shape

      x_flatten = x.view(-1, self.d_model)

      # route tokens to top-k experts
      scores = self.router(x_flatten)
      experts_scores, experts_ids = torch.topk(scores, self.top_k, dim=-1, sorted=False)
      experts_scores_softmax = torch.softmax(experts_scores, dim=-1)

      final_output = torch.zeros_like(x_flatten, device=x.device)

      # [B * S * top_k]
      token_ids_for_scatter = torch.arange(B * S, device=x.device).unsqueeze(-1).expand(-1, self.top_k).reshape(-1)

      experts_ids_flatten = experts_ids.view(-1)
      experts_scores_softmax_flatten = experts_scores_softmax.view(-1)

      for k in range(self.top_k):
        mask = (
            experts_ids_flatten == k
        )

        selected_tokens = x_flatten[token_ids_for_scatter[mask]]
        weights_for_expert = experts_scores_softmax_flatten[mask]

        weighted_expert_output = self.experts[k](selected_tokens) * weights_for_expert.unsqueeze(-1)

        final_output.scatter_add_(0, token_ids_for_scatter[mask].unsqueeze(-1).expand(-1, self.d_model), weighted_expert_output)

      return final_output.view(B, S, d_model)


In [8]:
# 🧪 Debug
moe = MixtureOfExperts(32, 64, num_experts=4, top_k=2)
x = torch.randn(2, 8, 32)
print('Output:', moe(x).shape)
print('Params:', sum(p.numel() for p in moe.parameters()))

torch.Size([8, 32])
torch.Size([8])
torch.Size([7, 32])
torch.Size([7])
Output: torch.Size([2, 8, 32])
Params: 16900


In [9]:
# ✅ SUBMIT
from torch_judge import check
check('moe')


🧪 Testing: Mixture of Experts (MoE) (Hard)
──────────────────────────────────────────────────
torch.Size([8, 32])
torch.Size([8])
torch.Size([7, 32])
torch.Size([7])
  ✅ [1/4] Output shape (2.9ms)
  ✅ [2/4] Has router and experts (1.3ms)
  ✅ [3/4] Router logits shape (3.8ms)
torch.Size([3, 16])
torch.Size([3])
torch.Size([3, 16])
torch.Size([3])
  ✅ [4/4] Gradient flow (49.6ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (57.5ms total)
  Progress saved. Run status() to see your dashboard.

